In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import EfficientNetB0
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications.efficientnet import preprocess_input


IMG_SIZE = 160
BATCH_SIZE = 16
EPOCHS = 15
LR = 1e-4

TRAIN_DIR = "Dataset/Train"
VAL_DIR = "Dataset/Validation"

# DATA GENERATORS

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(preprocessing_function=preprocess_input)


In [55]:
train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_data = val_gen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

Found 140002 images belonging to 2 classes.
Found 39428 images belonging to 2 classes.


In [ ]:
# MODEL:EfficientNet

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False  # freeze pretrained layers

In [57]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"


model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [58]:
model.summary()

Model: "model_8"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_8 (InputLayer)           [(None, 160, 160, 3  0           []                               
                                )]                                                                
                                                                                                  
 rescaling_14 (Rescaling)       (None, 160, 160, 3)  0           ['input_8[0][0]']                
                                                                                                  
 normalization_7 (Normalization  (None, 160, 160, 3)  7          ['rescaling_14[0][0]']           
 )                                                                                                
                                                                                            

In [ ]:
# CALLBACKS

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ModelCheckpoint("deepfake_efficientnet.h5",save_best_only=True,save_weights_only=True )
]

In [ ]:
# TRAIN

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=callbacks,
)


Epoch 1/15
8751/8751 [==============================] - 661s 75ms/step - loss: 0.4971 - accuracy: 0.7536 - val_loss: 0.4772 - val_accuracy: 0.7682
Epoch 2/15
8751/8751 [==============================] - 639s 73ms/step - loss: 0.4495 - accuracy: 0.7843 - val_loss: 0.4432 - val_accuracy: 0.7872
Epoch 3/15
8751/8751 [==============================] - 698s 80ms/step - loss: 0.4347 - accuracy: 0.7933 - val_loss: 0.4428 - val_accuracy: 0.7885
Epoch 4/15
8751/8751 [==============================] - 986s 113ms/step - loss: 0.4265 - accuracy: 0.7971 - val_loss: 0.4389 - val_accuracy: 0.7900
Epoch 5/15
8751/8751 [==============================] - 746s 85ms/step - loss: 0.4198 - accuracy: 0.8024 - val_loss: 0.4371 - val_accuracy: 0.7915
Epoch 6/15
8751/8751 [==============================] - 551s 63ms/step - loss: 0.4148 - accuracy: 0.8038 - val_loss: 0.4290 - val_accuracy: 0.7969
Epoch 7/15
8751/8751 [==============================] - 651s 74ms/step - loss: 0.4092 - accuracy: 0.8088 - val_loss: 

In [61]:
## fine tuning
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)


Epoch 1/5
8751/8751 [==============================] - 736s 84ms/step - loss: 0.4315 - accuracy: 0.8021 - val_loss: 0.3515 - val_accuracy: 0.8485
Epoch 2/5
8751/8751 [==============================] - 653s 75ms/step - loss: 0.3186 - accuracy: 0.8603 - val_loss: 0.3041 - val_accuracy: 0.8736
Epoch 3/5
8751/8751 [==============================] - 660s 75ms/step - loss: 0.2842 - accuracy: 0.8781 - val_loss: 0.2759 - val_accuracy: 0.8879
Epoch 4/5
8751/8751 [==============================] - 637s 73ms/step - loss: 0.2601 - accuracy: 0.8894 - val_loss: 0.2553 - val_accuracy: 0.8977
Epoch 5/5
8751/8751 [==============================] - 637s 73ms/step - loss: 0.2445 - accuracy: 0.8985 - val_loss: 0.2462 - val_accuracy: 0.9040


In [64]:
model.save_weights("deepfake_weights.h5")


In [77]:
## test data
test_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

test_data = test_gen.flow_from_directory(
    "Dataset/Test",
    target_size=(160,160),
    batch_size=16,
    class_mode='binary'
)

loss, acc = model.evaluate(test_data)
print("Test Accuracy:", acc)


Found 10905 images belonging to 2 classes.
682/682 [==============================] - 14s 21ms/step - loss: 0.5501 - accuracy: 0.7956
Test Accuracy: 0.7955983281135559
